# Bonus C — Agentic RAG

**When to use this module:** After Module 6 (Evaluation), if the group has time.

**What you'll learn:** The difference between standard and agentic RAG, how to give an LLM a retrieval tool it can call on its own, and how to implement a basic agent loop using Ollama's tool-calling API.

**Time:** ~45 minutes

---

## The limitation of standard RAG

Every RAG pipeline you've built so far follows the same fixed sequence:

```
user query → retrieve → generate → answer
```

This works well for straightforward questions. But it breaks down in two situations:

1. **The retrieval wasn't good enough.** The LLM gets weak context and either hallucinates or says 'I don't know' — but it has no way to try again.
2. **The question requires multiple retrievals.** _'Compare the sick leave policy to the parental leave policy'_ needs two separate lookups. A single retrieve step will blend them into one noisy chunk.

Agentic RAG solves both by giving the LLM a **retrieval tool** it can call when it decides it needs information — as many times as it wants, with whatever query it thinks is most appropriate.

## The agent loop

Standard RAG is a **single-pass** system: one retrieve, one generate.
Agentic RAG turns this into a **loop** where the LLM decides when it has
enough information to answer, and can request more if not.

```
User question
    |
    v
┌───────────────────────────────┐
│  LLM (with tools)             │
│                               │
│  "I need to look up X"  ──────┼──► retrieve(X) ──► context added
│  "I need to look up Y"  ──────┼──► retrieve(Y) ──► context added
│  "I have enough to answer"    │
└───────────────────────────────┘
    |
    v
Final answer
```

This pattern is called **ReAct** (Reason + Act): the LLM alternates between
reasoning about what it needs and taking actions (tool calls) to get it.

### Tool calling: how it works technically

Modern LLMs (including Llama 3.x via Ollama) support **tool calling** natively.
You describe your tools in a JSON schema, and the model can output structured
JSON specifying which tool to call with which arguments.

```json
{
  "name": "search_documents",
  "arguments": {
    "query": "parental leave policy duration"
  }
}
```

Your code detects this output, runs the actual search, and returns the result
to the model in the next turn. The model then decides whether to call another
tool or generate the final answer.

### When agentic RAG is worth the complexity

Agentic RAG adds latency (multiple LLM calls) and complexity (tool parsing, loop logic).
It is worth it when:
- Questions require information from multiple distinct parts of the corpus
- You cannot predict in advance how many retrievals are needed
- The retrieval strategy should depend on what was found (adaptive search)

For simple Q&A over a small, well-structured corpus, standard RAG is sufficient
and much easier to debug.

### Failure modes unique to agentic RAG

- **Infinite loops** — the model keeps calling the tool without converging (always add a max-steps limit)
- **Tool hallucination** — the model invents tool calls with arguments it made up
- **Context accumulation** — after many retrievals, the context window fills up (need to summarise)


In [ ]:
import json
import requests
import chromadb
from chromadb.utils import embedding_functions

from ragsst.utils import list_files, read_file, get_chunks
from ragsst.parameters import EMBEDDING_MODEL, LLMBASEURL, MODEL, DATA_PATH

print('Imports ready.')

---

## Part 1 — Build the vector store

Same as Module 4 — nothing new here.

In [ ]:
client = chromadb.Client()
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL
)
collection = client.get_or_create_collection(
    name='agentic_demo',
    embedding_function=embedding_func,
    metadata={'hnsw:space': 'cosine'},
)

files = list_files(DATA_PATH, extensions=('.txt', '.pdf'))
for f in files:
    text = read_file(f)
    chunks = get_chunks(text)
    collection.add(
        documents=chunks,
        ids=[f'chunk_{f}_{i}' for i in range(len(chunks))],
    )

print(f'Vector store ready: {collection.count()} chunks.')

---

## Part 2 — Define the retrieval tool

Ollama supports tool calling using the same JSON schema format as OpenAI. We define a tool as a Python function plus a description that tells the LLM when and how to use it.

The description is critical — the LLM reads it to decide whether to call the tool. Write it as if you're explaining the tool to a colleague.

In [ ]:
# The actual Python function that does the retrieval
def search_documents(query: str, n_results: int = 3) -> str:
    """
    Search the document corpus for chunks relevant to the query.
    Returns the retrieved text as a single string.
    """
    results = collection.query(query_texts=[query], n_results=n_results)
    chunks = results['documents'][0]
    return '\n\n---\n\n'.join(chunks)


# The JSON schema description the LLM sees
TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'search_documents',
            'description': (
                'Search the local document corpus for information relevant to a query. '
                'Use this tool whenever you need factual information to answer a question. '
                'You can call it multiple times with different queries if needed.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {
                        'type': 'string',
                        'description': 'The search query. Be specific — a focused query retrieves better results than a broad one.',
                    },
                    'n_results': {
                        'type': 'integer',
                        'description': 'Number of document chunks to retrieve. Default 3. Use 5 for complex topics.',
                        'default': 3,
                    },
                },
                'required': ['query'],
            },
        },
    }
]

# Map tool names to Python functions
TOOL_REGISTRY = {
    'search_documents': search_documents,
}

print('Tool defined.')

---

## Part 3 — The agent loop

The agent loop is the core of agentic RAG. It works like this:

1. Send the user's question to the LLM, along with the tool definition
2. If the LLM responds with a tool call, execute the function and send the result back
3. Repeat until the LLM gives a plain text answer (no tool call)
4. Return that answer

The LLM decides when it has enough information. It might call the tool once, twice, or not at all.

In [ ]:
def call_llm(messages: list[dict], tools: list[dict] | None = None) -> dict:
    """Send messages to Ollama and return the raw response dict."""
    url = LLMBASEURL + '/chat'
    payload = {
        'model': MODEL,
        'messages': messages,
        'stream': False,
    }
    if tools:
        payload['tools'] = tools

    r = requests.post(url, json=payload)
    return json.loads(r.text)


def agentic_rag(user_question: str, max_iterations: int = 5, verbose: bool = True) -> str:
    """
    Answer a question using an agentic RAG loop.

    The LLM can call search_documents as many times as it needs
    before producing a final answer.
    """
    messages = [
        {
            'role': 'system',
            'content': (
                'You are a helpful assistant with access to a document search tool. '
                'Use the tool to retrieve relevant information before answering. '
                'If the first search does not give you enough information, search again with a different query. '
                'Be concise and base your answer only on the retrieved documents.'
            ),
        },
        {
            'role': 'user',
            'content': user_question,
        },
    ]

    for iteration in range(max_iterations):
        if verbose:
            print(f'[Iteration {iteration + 1}] Calling LLM...')

        response = call_llm(messages, tools=TOOLS)
        message = response.get('message', {})
        tool_calls = message.get('tool_calls', [])

        # No tool calls — the LLM is ready to answer
        if not tool_calls:
            final_answer = message.get('content', '')
            if verbose:
                print(f'[Iteration {iteration + 1}] LLM produced final answer.')
            return final_answer

        # Append the LLM's message (with tool calls) to history
        messages.append({'role': 'assistant', 'content': None, 'tool_calls': tool_calls})

        # Execute each tool call and append results
        for tool_call in tool_calls:
            fn_name = tool_call['function']['name']
            fn_args = tool_call['function'].get('arguments', {})

            if verbose:
                print(f'[Iteration {iteration + 1}] Tool call: {fn_name}({fn_args})')

            if fn_name in TOOL_REGISTRY:
                result = TOOL_REGISTRY[fn_name](**fn_args)
            else:
                result = f'Error: unknown tool "{fn_name}"'

            if verbose:
                print(f'[Iteration {iteration + 1}] Retrieved {len(result)} characters.')

            messages.append({
                'role': 'tool',
                'content': result,
            })

    return 'Max iterations reached without a final answer.'

---

## Part 4 — Run the agent

In [ ]:
# A simple question — the agent should call the tool once
question = 'What AI services are available in Berlin?'
print(f'Question: {question}\n')

answer = agentic_rag(question, verbose=True)
print(f'\nFinal answer:\n{answer}')

In [ ]:
# A question that might need multiple retrievals
# The agent should search once for each topic, then synthesise
question = 'Who is Sherlock Holmes and what does Die Hard have to do with Christmas?'
print(f'Question: {question}\n')

answer = agentic_rag(question, verbose=True)
print(f'\nFinal answer:\n{answer}')

**Observe:** Watch the iteration log. Does the LLM call the tool twice for the second question? Does it formulate different queries for each topic? This is the key difference from standard RAG — the LLM is *driving* the retrieval, not just receiving it.

**To do:**
1. Ask a question where the first retrieval is likely to be insufficient. Does the agent recover?
2. Set `verbose=True` and read the tool call arguments. Are the LLM's search queries better or worse than the original user question?
3. Try lowering `max_iterations=1`. What changes?
4. Add a second tool — for example, `get_document_list()` that returns the filenames in the corpus. Does the LLM use it to orient itself before searching?

---

## Part 5 — Reflection

Standard RAG and agentic RAG are tools for different situations:

| | Standard RAG | Agentic RAG |
|---|---|---|
| Retrieval | Once, before generation | Multiple times, on demand |
| Query | User's original question | LLM-generated, refined |
| Latency | Fast | Slower (multiple LLM calls) |
| Complexity | Low | Higher |
| Best for | Simple, focused questions | Multi-part, exploratory queries |

Agentic RAG is more powerful but also more expensive and harder to debug. For most production use cases, well-tuned standard RAG with re-ranking (Module 5) will outperform a basic agent at lower cost. Agents shine when the question structure is genuinely unpredictable.

The pattern you've implemented here — a tool-equipped LLM in a loop — is the foundation of more complex agent frameworks like LangGraph, AutoGen, and CrewAI.

---

## Further reading

- [Ollama tool calling docs](https://ollama.com/blog/tool-support)
- [Agentic RAG survey (arXiv)](https://arxiv.org/abs/2401.15884)
- [LangGraph — stateful agent framework](https://langchain-ai.github.io/langgraph/)
- [ReAct: reasoning and acting in LLMs](https://arxiv.org/abs/2210.03629)